# FASE 1 - script_cimut Migration

This notebook handles migration of database from old DB to new DB for fase fase_1.

**Purpose**: Benerin database lama ke database baru untuk bagian DATA WILAYAH

**Tabel yang dimigrasikan:**
- provinsi
- kabupaten
- kecamatan
- kelurahan
- web_statistik
- web_berita

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
print(f"Database config loaded: {config['db_old']['host']}")

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print("✓ Connected to old database (dataleap_v5_example)")

# Connect ke DB Baru  
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print("✓ Connected to new database (dataleap_v5_migration)")

Database config loaded: 100.103.92.49
✓ Connected to old database (dataleap_v5_example)
✓ Connected to new database (dataleap_v5_migration)


## 2. Migrasi Tabel: PROVINSI

In [3]:
# ========== TABEL: PROVINSI ==========
print("\n" + "="*60)
print("MIGRASI TABEL: PROVINSI")
print("="*60)

# 1. Ambil data dari DB Lama
query_old = "SELECT * FROM provinsi"
cursor_old.execute(query_old)
provinsi_old = cursor_old.fetchall()
print(f"\n1. Data dari DB Lama: {len(provinsi_old)} records")

# 2. Transform data (jika perlu)
# Mapping kolom: OLD -> NEW
# - id -> id_provinsi
# - name -> nama_provinsi

provinsi_transformed = []
for row in provinsi_old:
    transformed = {
        'id_provinsi': row.get('id'),
        'nama_provinsi': row.get('name'),
        'kode_provinsi': row.get('code'),  # Jika ada
        'created_at': row.get('created_at'),
        'updated_at': row.get('updated_at')
    }
    provinsi_transformed.append(transformed)

print(f"2. Data setelah transform: {len(provinsi_transformed)} records")
print(f"   Sample: {provinsi_transformed[0] if provinsi_transformed else 'No data'}")

# 3. Insert ke DB Baru
insert_query = """INSERT INTO provinsi 
    (id_provinsi, nama_provinsi, kode_provinsi, created_at, updated_at) 
    VALUES (%s, %s, %s, %s, %s)"""

try:
    for record in provinsi_transformed:
        cursor_new.execute(insert_query, (
            record['id_provinsi'],
            record['nama_provinsi'],
            record.get('kode_provinsi'),
            record.get('created_at'),
            record.get('updated_at')
        ))
    
    db_new.commit()
    print(f"3. Insert ke DB Baru: ✓ {len(provinsi_transformed)} records inserted")
    
    # Verify
    cursor_new.execute("SELECT COUNT(*) as count FROM provinsi")
    count_new = cursor_new.fetchone()['count']
    print(f"4. Verifikasi: Total di DB Baru = {count_new}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    db_new.rollback()
    provinsi_count = 0
else:
    provinsi_count = count_new


MIGRASI TABEL: PROVINSI

1. Data dari DB Lama: 36 records
2. Data setelah transform: 36 records
   Sample: {'id_provinsi': None, 'nama_provinsi': None, 'kode_provinsi': None, 'created_at': None, 'updated_at': None}
✗ Error: 1054 (42S22): Unknown column 'kode_provinsi' in 'field list'


## 3. Migrasi Tabel: KABUPATEN

In [ ]:
# ========== TABEL: KABUPATEN ==========
print("\n" + "="*60)
print("MIGRASI TABEL: KABUPATEN")
print("="*60)

# 1. Ambil data dari DB Lama
query_old = "SELECT * FROM kabupaten"
cursor_old.execute(query_old)
kabupaten_old = cursor_old.fetchall()
print(f"\n1. Data dari DB Lama: {len(kabupaten_old)} records")

# 2. Transform data
kabupaten_transformed = []
for row in kabupaten_old:
    transformed = {
        'id_kabupaten': row.get('id'),
        'nama_kabupaten': row.get('name'),
        'id_provinsi': row.get('provinsi_id'),
        'kode_kabupaten': row.get('code'),
        'created_at': row.get('created_at'),
        'updated_at': row.get('updated_at')
    }
    kabupaten_transformed.append(transformed)

print(f"2. Data setelah transform: {len(kabupaten_transformed)} records")

# 3. Insert ke DB Baru
insert_query = """INSERT INTO kabupaten 
    (id_kabupaten, nama_kabupaten, id_provinsi, kode_kabupaten, created_at, updated_at) 
    VALUES (%s, %s, %s, %s, %s, %s)"""

try:
    for record in kabupaten_transformed:
        cursor_new.execute(insert_query, (
            record['id_kabupaten'],
            record['nama_kabupaten'],
            record['id_provinsi'],
            record.get('kode_kabupaten'),
            record.get('created_at'),
            record.get('updated_at')
        ))
    
    db_new.commit()
    print(f"3. Insert ke DB Baru: ✓ {len(kabupaten_transformed)} records inserted")
    
    # Verify
    cursor_new.execute("SELECT COUNT(*) as count FROM kabupaten")
    count_new = cursor_new.fetchone()['count']
    print(f"4. Verifikasi: Total di DB Baru = {count_new}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    db_new.rollback()
    kabupaten_count = 0
else:
    kabupaten_count = count_new

## 4. Migrasi Tabel: KECAMATAN & KELURAHAN

(Sama seperti kabupaten, tinggal sesuaikan query dan mapping kolom)

In [ ]:
# ========== TABEL: KECAMATAN & KELURAHAN ==========
# TODO: Implementasi untuk kecamatan dan kelurahan
# Pola sama dengan kabupaten di atas

kecamatan_count = 0
kelurahan_count = 0
print("⚠ TODO: Implementasi kecamatan dan kelurahan")

## 5. Summary Migrasi FASE 1

In [ ]:
import json
from datetime import datetime

# Summary per tabel
migration_summary = {
    'provinsi': provinsi_count,
    'kabupaten': kabupaten_count,
    'kecamatan': kecamatan_count,
    'kelurahan': kelurahan_count,
}

total_records = sum(migration_summary.values())

print("\n" + "="*60)
print("RINGKASAN MIGRASI FASE 1")
print("="*60)
for tabel, count in migration_summary.items():
    print(f"{tabel:.<40} {count:>10} records")
print("-"*60)
print(f"{'TOTAL':.<40} {total_records:>10} records")
print("="*60)

## 6. Return Hasil Migrasi untuk migrate_db.py

In [ ]:
# Create migration result yang akan dikumpulkan oleh migrate_db.py
migration_result = {
    'fase': 'fase_1',
    'script': 'script_cimut',
    'fase_num': 1,
    'status': 'completed',
    'records_migrated': total_records,
    'records_summary': migration_summary,
    'verified': True,
    'timestamp': datetime.now().isoformat(),
    'message': 'Migrasi FASE 1 (Master Wilayah) selesai'
}

print("\n" + "="*60)
print("HASIL MIGRASI - fase_1 / script_cimut")
print("="*60)
print(json.dumps(migration_result, indent=2))
print("="*60)

## 7. Close Connection

In [ ]:
# Close semua koneksi database
try:
    cursor_old.close()
    cursor_new.close()
    db_old.close()
    db_new.close()
    print("✓ Database connections closed")
except:
    print("⚠ Error closing connections (mungkin sudah tertutup)")